<a href="https://colab.research.google.com/github/Ladecodes1/formative-3-group/blob/main/part2_bayes_imdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Bayesian Probability on IMDb Movie Reviews
Goal: for each keyword, compute P(Positive | keyword) using Bayes' theorem.
Dataset: 50,000 IMDb reviews (25,000 positive, 25,000 negative).
Keywords: excellent, wonderful, superb, brilliant (positive); worst, awful, waste, boring (negative).

## Step 1: Load the data
We read the CSV with Python's built-in csv module. Each review is turned into a set of lowercase words, so a word is counted once per review.

In [ ]:
import csv
import re

DATA_PATH = "IMDB Dataset.csv"


def load_reviews(path):
    """Read the CSV and return a list of (set_of_words, label) pairs."""
    reviews = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            text = row["review"].lower().replace("<br />", " ")
            words = set(re.findall(r"[a-z']+", text))
            reviews.append((words, row["sentiment"].strip().lower()))
    return reviews


reviews = load_reviews(DATA_PATH)
print("Total reviews:", len(reviews))
print("Example label:", reviews[0][1])
print("Some words from review 1:", list(reviews[0][0])[:8])

Total reviews: 50000
Example label: positive
Some words from review 1: ['get', 'main', 'scuffles', 'developed', 'is', 'guards', 'inwards', 'hardcore']


## Step 2: Counting helper and the prior
One reusable function counts reviews by keyword and/or label. The prior P(Positive) is positive reviews ÷ total reviews.

In [ ]:
def count_reviews(reviews, keyword=None, label=None):
    """Count reviews, optionally only those containing a keyword and/or with a given label."""
    return sum(
        1
        for words, lab in reviews
        if (keyword is None or keyword in words)
        and (label is None or lab == label)
    )


total = count_reviews(reviews)
n_positive = count_reviews(reviews, label="positive")
prior = n_positive / total

print("Total reviews:", total)
print("Positive reviews:", n_positive)
print("Prior P(Positive):", prior)

Total reviews: 50000
Positive reviews: 25000
Prior P(Positive): 0.5


## Step 3: Bayes' theorem as a function, worked through for one keyword
P(Positive | keyword) = P(keyword | Positive) × P(Positive) ÷ P(keyword)

`analyse_keyword` builds all four probabilities from `count_reviews`, so the same function gives the worked example here and the full table in Step 4.

In [ ]:
def bayes_posterior(prior, likelihood, marginal):
    """Bayes' theorem: P(A|B) = P(B|A) * P(A) / P(B)"""
    return likelihood * prior / marginal


def analyse_keyword(reviews, keyword):
    """Return the counts and the four probabilities for P(Positive | keyword)."""
    total = count_reviews(reviews)
    n_positive = count_reviews(reviews, label="positive")
    n_keyword = count_reviews(reviews, keyword=keyword)
    n_both = count_reviews(reviews, keyword=keyword, label="positive")

    prior = n_positive / total
    likelihood = n_both / n_positive
    marginal = n_keyword / total
    posterior = bayes_posterior(prior, likelihood, marginal)

    # Sanity check: Bayes must match the direct count
    assert abs(posterior - n_both / n_keyword) < 1e-9

    return {
        "keyword": keyword,
        "n_keyword": n_keyword,
        "n_both": n_both,
        "prior": prior,
        "likelihood": likelihood,
        "marginal": marginal,
        "posterior": posterior,
    }


example = analyse_keyword(reviews, "excellent")
kw = example["keyword"]

print("Keyword:", kw)
print("Reviews containing it:", example["n_keyword"])
print("Of those, positive:", example["n_both"])
print()
print(f"Prior      P(Positive)             = {example['prior']}")
print(f"Likelihood P({kw} | Positive) = {example['likelihood']}")
print(f"Marginal   P({kw})            = {example['marginal']}")
print(f"Posterior  P(Positive | {kw}) = {example['posterior']}")


## Step 4: All eight keywords

In [ ]:
POSITIVE_KEYWORDS = ["excellent", "wonderful", "superb", "brilliant"]
NEGATIVE_KEYWORDS = ["worst", "awful", "waste", "boring"]

results = [analyse_keyword(reviews, k) for k in POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS]

print(f"{'Keyword':<12}{'P(Pos)':>9}{'P(kw|Pos)':>12}{'P(kw)':>9}{'P(Pos|kw)':>12}")
for r in results:
    print(f"{r['keyword']:<12}{r['prior']:>9.4f}{r['likelihood']:>12.4f}"
          f"{r['marginal']:>9.4f}{r['posterior']:>12.4f}")


## Step 5: Posterior for each keyword
The dashed line is the prior. A bar to its right means the keyword makes a positive review more likely; a bar to its left means less likely.

In [ ]:
import matplotlib.pyplot as plt

POSITIVE_COLOR = "#2a9d8f"
NEGATIVE_COLOR = "#e76f51"


def plot_posteriors(results, positive_keywords):
    """Horizontal bar chart of P(Positive | keyword), sorted, with the prior marked."""
    ordered = sorted(results, key=lambda r: r["posterior"])
    names = [r["keyword"] for r in ordered]
    values = [r["posterior"] for r in ordered]
    colors = [POSITIVE_COLOR if n in positive_keywords else NEGATIVE_COLOR for n in names]
    prior = ordered[0]["prior"]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(names, values, color=colors)
    for y, v in enumerate(values):
        ax.annotate(f"{v:.3f}", (v, y), xytext=(5, 0), textcoords="offset points", va="center")
    ax.axvline(prior, color="#555555", linestyle="--", linewidth=1)
    ax.annotate(f"prior = {prior:.1f}", (prior, 0), xytext=(5, 0),
                textcoords="offset points", va="center", color="#555555")
    ax.set_xlim(0, 1)
    ax.set_xlabel("P(Positive | keyword)")
    ax.set_title("Positive keywords lift the posterior above the prior; negative ones pull it below")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    return fig


plot_posteriors(results, POSITIVE_KEYWORDS)
plt.show()


## Interpretation
Positive keywords all have posteriors well above 0.5 (superb is strongest at 0.856, brilliant weakest at 0.761).
Negative keywords all have posteriors well below 0.5 (waste is lowest at 0.069, boring highest at 0.194), so reviews containing them are unlikely to be positive.